# Weather Report: Climate Change and Cyclone Ditwah in Sri Lanka

**Research Question:** Is Sri Lanka's climate changing, and was Cyclone Ditwah (26-30 Nov 2025) a symptom of that trend or a statistical outlier?

### Data Sources & Architecture
Data is sourced from the Open-Meteo Historical Weather API. It consists of ~31,000 days of reanalysis grid data (1940-2025) across 10 major locations in Sri Lanka.

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Fix ModuleNotFoundError: Add the src/ folder to Python's search path
src_path = str(Path.cwd().parent / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# Import our custom project modules
from ditwah_weather.io import load_all
from ditwah_weather.config import LOCATIONS

# Load all 10 cities into one dataset
data_dir = Path.cwd().parent / "data"
df = load_all(data_dir)

print(f"Successfully loaded {len(df)} rows across {df['city'].nunique()} grid points.")
df.head(3)

Successfully loaded 314120 rows across 10 grid points.


,time,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max,date,location,city
0,1940-01-01,29.6,22.7,NaN,18.8,1940-01-01,Negombo,Negombo
1,1940-01-02,30.7,23.1,0.0,18.8,1940-01-02,Negombo,Negombo
2,1940-01-03,30.4,22.5,0.0,18.0,1940-01-03,Negombo,Negombo


## Act 1: The Warming Trend
Before looking at extreme rainfall events, we must establish if the background climate in Sri Lanka is changing. We calculate the annual maximum temperature anomalies (comparing each year to a standard 1961-1990 baseline) for all 10 grid points.

In [3]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
from ditwah_weather.analysis import add_anomalies, annual_anomaly, decadal_trend

# 1. Calculate daily anomalies for all data
df_anom = add_anomalies(df, base=(1961, 1990), var='temperature_2m_max')

# 2. Calculate the warming trend per decade for all 10 cities
trends = []
city_annual_series = {}

for city in df_anom['city'].unique():
    # Filter the daily data for just one city
    city_df = df_anom[df_anom['city'] == city]
    
    # Calculate the annual average anomaly for this specific city
    # annual_anomaly() returns a pandas Series indexed by 'year'
    city_annual = annual_anomaly(city_df)
    city_annual_series[city] = city_annual
    
    # Calculate the trend
    slope, p_val = decadal_trend(city_annual.index, city_annual.values)
    trends.append({'City': city, 'Trend (°C / decade)': round(slope, 3), 'p-value': round(p_val, 4)})

# Display the results table
trend_df = pd.DataFrame(trends).sort_values('Trend (°C / decade)', ascending=False)
display(trend_df)

# 3. Visualisation: Anomaly Time Series for Dankotuwa
dankotuwa_annual = city_annual_series['Dankotuwa'].reset_index()
dankotuwa_annual['Rolling_10yr'] = dankotuwa_annual['anomaly'].rolling(10, center=True).mean()

fig = px.bar(dankotuwa_annual, x='year', y='anomaly', 
             title='Annual Temperature Anomaly in Dankotuwa (Baseline: 1961-1990)',
             labels={'year': 'Year', 'anomaly': 'Temperature Anomaly (°C)'},
             color='anomaly', color_continuous_scale='RdBu_r')

# Add the 10-year rolling mean as a trendline
fig.add_trace(go.Scatter(x=dankotuwa_annual['year'], y=dankotuwa_annual['Rolling_10yr'],
                         mode='lines', name='10-Year Rolling Mean',
                         line=dict(color='black', width=3)))

fig.update_layout(template='plotly_white', coloraxis_showscale=False)
fig.show()

,City,Trend (°C / decade),p-value
7,Trincomalee,0.152,0.0000
4,Nuwara_eliya,0.106,0.0065
5,Anuradhapura,0.095,0.0010
3,Ratnapura,0.094,0.0000
8,Galle,0.087,0.0000
9,Dankotuwa,0.061,0.0002
0,Negombo,0.058,0.0004
6,Batticaloa,0.056,0.0030
2,Kandy,0.051,0.0843
1,Colombo,0.043,0.0073


## Act 2: Rain Extremes
While the temperature is clearly rising, we must check if extreme rainfall is also increasing. We analyze two metrics for the baseline period (1940-2024):
1. The trend in the maximum 5-day rainfall each November.
2. The trend in the number of "heavy rain" days (over 50 mm) per year.
Finally, we plot Dankotuwa's historical 5-day November maxima and highlight Cyclone Ditwah (2025) to visually assess if it was a statistical outlier.

In [4]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from ditwah_weather.analysis import annual_max_window, decadal_trend, calculate_event_rainfall

# 1. Calculate trends for November 5-Day Max Rainfall & Heavy Rain Days (>50mm)
rain_trends = []

for city in df['city'].unique():
    city_df = df[df['city'] == city].copy()
    
    # --- Metric 1: 5-Day Max Rainfall (November, 1940-2024) ---
    annual_max = annual_max_window(city_df, window=5, month=11)
    max_slope, max_pval = decadal_trend(annual_max.index, annual_max.values)
    
    # --- Metric 2: Heavy Rain Days (Whole Year, 1940-2024) ---
    # Filter for the baseline years
    baseline_df = city_df[(city_df['date'].dt.year >= 1940) & (city_df['date'].dt.year <= 2024)]
    # Count days > 50mm, then reindex to ensure years with 0 heavy days are included as 0
    heavy_days = baseline_df[baseline_df['precipitation_sum'] > 50].groupby(baseline_df['date'].dt.year).size()
    heavy_days = heavy_days.reindex(range(1940, 2025), fill_value=0) 
    heavy_slope, heavy_pval = decadal_trend(heavy_days.index, heavy_days.values)
    
    rain_trends.append({
        'City': city, 
        '5-Day Max Trend (mm/decade)': round(max_slope, 2), 'Max p-val': round(max_pval, 4),
        'Heavy Days Trend (days/decade)': round(heavy_slope, 3), 'Heavy p-val': round(heavy_pval, 4)
    })

    # Save Dankotuwa's historical max data for plotting below
    if city == 'Dankotuwa':
        dankotuwa_max_rain = annual_max.reset_index()

# Display the trends table
display(pd.DataFrame(rain_trends).sort_values('5-Day Max Trend (mm/decade)', ascending=False))


# 2. Calculate exact rainfall for Cyclone Ditwah (2025) in Dankotuwa
dankotuwa_df = df[df['city'] == 'Dankotuwa']
ditwah_total = calculate_event_rainfall(dankotuwa_df, start='2025-11-26', end='2025-11-30')

# 3. Visualisation: Bar chart of historical max vs Ditwah
fig = px.bar(dankotuwa_max_rain, x='year', y='precipitation_sum',
             title='Annual Maximum 5-Day Rainfall in November (Dankotuwa)',
             labels={'year': 'Year', 'precipitation_sum': '5-Day Total Rainfall (mm)'})

# Update historical bars to standard blue
fig.update_traces(marker_color='#1f77b4', name='Historical Max (1940-2024)', showlegend=True)

# Add Cyclone Ditwah as a striking red bar
fig.add_trace(go.Bar(x=[2025], y=[ditwah_total], name='Cyclone Ditwah (Nov 2025)', marker_color='red'))

fig.update_layout(template='plotly_white', legend=dict(x=0.02, y=0.98))
fig.show()

,City,5-Day Max Trend (mm/decade),Max p-val,Heavy Days Trend (days/decade),Heavy p-val
7,Trincomalee,9.79,0.0001,0.221,0.0011
0,Negombo,8.56,0.0001,0.244,0.0009
9,Dankotuwa,8.39,0.0001,0.229,0.0016
5,Anuradhapura,8.38,0.0001,0.156,0.0009
8,Galle,8.08,0.0000,0.167,0.0074
1,Colombo,7.61,0.0007,0.249,0.0019
6,Batticaloa,7.45,0.0008,0.269,0.0003
2,Kandy,5.23,0.0086,0.175,0.0052
4,Nuwara_eliya,4.70,0.0066,0.166,0.0175
3,Ratnapura,2.47,0.4718,0.428,0.0468


## Act 3: Cyclone Ditwah in Context
We analyze the footprint of Cyclone Ditwah (26-30 Nov 2025) across all 10 grid points. 
We compare the 5-day event rainfall against the 1940-2024 baseline to calculate its historical percentile. Finally, we fit a Generalized Extreme Value (GEV) distribution to the historical annual maxima to estimate the **Return Period**. 

*Caveat:* With only 85 data points and coarse reanalysis grid data, return periods for unprecedented events are highly uncertain. They should be interpreted as an order of magnitude rather than an exact forecast.

In [7]:
import pandas as pd
import plotly.express as px
from ditwah_weather.analysis import calculate_event_rainfall, calculate_historical_percentile, annual_max_window, return_period
from ditwah_weather.config import LOCATIONS

act3_results = []
for city in df['city'].unique():
    city_df = df[df['city'] == city].copy()
    
    # 1. Event Rainfall (Nov 26-30, 2025)
    event_rain = calculate_event_rainfall(city_df, start='2025-11-26', end='2025-11-30')
    
    # 2. Historical Percentile (Compared to all historical 5-day November windows)
    percentile = calculate_historical_percentile(
        city_df, event_total=event_rain, window=5, month=11, 
        first_year=1940, last_year=2024
    )
    
    # 3. GEV Return Period
    hist_max = annual_max_window(city_df, window=5, month=11, first_year=1940, last_year=2024)
    rp = return_period(hist_max.values, event_rain)
    
    # 4. Wind Speed (Max during event)
    event_mask = (city_df['date'] >= '2025-11-26') & (city_df['date'] <= '2025-11-30')
    event_wind = city_df.loc[event_mask, 'wind_speed_10m_max'].max()
    
    # Format Return Period safely (order of magnitude)
    if rp == float('inf') or rp > 10000:
        rp_str = "> 10,000 years"
    else:
        # Round to the nearest 10 or 100 to avoid false precision
        rp_str = f"~{int(round(rp, -1)):,} years"
        
    act3_results.append({
        'City': city,
        'Ditwah Rain (mm)': round(event_rain, 1),
        'Max Wind (km/h)': round(event_wind, 1),
        'Historical Percentile': f"{percentile}%",
        'Estimated Return Period': rp_str
    })

# Display summary table
act3_df = pd.DataFrame(act3_results).sort_values('Ditwah Rain (mm)', ascending=False)
display(act3_df)

# 5. Visualisation: Map of the Cyclone's Footprint
# Extract coordinates from config.py
coords = pd.DataFrame([{'City': loc.name, 'lat': loc.latitude, 'lon': loc.longitude} for loc in LOCATIONS])
map_df = pd.merge(act3_df, coords, on='City')

fig = px.scatter_map(
    map_df, lat="lat", lon="lon", size="Ditwah Rain (mm)", color="Ditwah Rain (mm)",
    hover_name="City", hover_data=["Estimated Return Period", "Max Wind (km/h)"],
    color_continuous_scale="Blues", zoom=6.5, center={"lat": 7.8, "lon": 80.7},
    title="Cyclone Ditwah: 5-Day Rainfall Distribution (Grid Points)"
)
# Note: mapbox_style is also renamed to map_style
fig.update_layout(map_style="carto-positron", margin={"r":0,"t":40,"l":0,"b":0})
fig.show()

,City,Ditwah Rain (mm),Max Wind (km/h),Historical Percentile,Estimated Return Period
0,Negombo,570.7,48.1,100.0%,"> 10,000 years"
5,Anuradhapura,520.3,45.5,100.0%,"~8,320 years"
9,Dankotuwa,480.6,47.9,100.0%,"> 10,000 years"
7,Trincomalee,389.0,58.8,99.95%,~750 years
2,Kandy,371.1,27.6,100.0%,"> 10,000 years"
4,Nuwara_eliya,362.7,38.7,100.0%,"> 10,000 years"
1,Colombo,351.2,41.9,100.0%,"> 10,000 years"
3,Ratnapura,281.7,30.9,98.73%,~10 years
6,Batticaloa,199.9,46.3,99.46%,~20 years
8,Galle,179.7,34.5,99.5%,~10 years
